# End-to-End Performance (Section 4.2)

This notebook generates the first-part experiment artifacts for **FedAvg-Full**, **FedMoE**, **FLEx**, and **FedM$^2$oE**:

- **Figure 1**: Accuracy vs. Communication Rounds
- **Figure 2**: Accuracy vs. Wall-clock Time
- **Main Results Table**: final accuracy/time/speedup summary

Setting: NIID with Dirichlet $\alpha=0.5$, simulated 8 clients.


In [ ]:
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Plot style aligned with notebook_example.ipynb
PAPER_RC: Dict[str, object] = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linestyle": ":",
    "grid.alpha": 0.4,
    "lines.linewidth": 2.0,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

sns.set_theme(style="whitegrid", rc=PAPER_RC)

METHOD_COLORS: Dict[str, str] = {"full": "#E15759", "mix": "#4E79A7", "drop": "#59A14F", "flex": "#F28E2B"}

METHOD_MARKERS: Dict[str, str] = {"full": "o", "mix": "s", "drop": "^", "flex": "D"}

METHOD_LABELS: Dict[str, str] = {"full": "FedAvg-Full", "mix": "Fed-M$^2$oE (Ours)", "drop": "FedMoE", "flex": "FLEx"}

DATASET_LABELS: Dict[str, str] = {
    "20news": "20Newsgroups",
    "agnews": "AG News",
    "emotion": "Emotion",
}

RESULTS_BASE = Path("../../outputs/NIID/main_exp")
FIG_DIR = RESULTS_BASE / "figures"
TABLE_DIR = RESULTS_BASE / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ORDER: List[str] = ["20news", "agnews", "emotion"]
MODE_ORDER: List[str] = ["full", "mix", "drop", "flex"]

# ------------------------------
# Communication-time hyper-parameters
# ------------------------------
# Effective bandwidths (MB/s). We model synchronous FL transfer time as
# download_size / downlink + upload_size / uplink for each completed round.
DOWNLINK_MBPS: float = 50
UPLINK_MBPS: float = 20

# Communication volume per client per FL round (MB), from profile_memory_communication.ipynb.
DOWNLOAD_SIZE_MB: Dict[str, float] = {
    "full": 2550.852554321289,
    "mix": 2118.852554321289,
    "drop": 1686.711929321289,
    "flex": 1038.606460571289,
}
UPLOAD_SIZE_MB: Dict[str, float] = {
    "full": 2550.852554321289,
    "mix": 1686.852554321289,
    "drop": 1686.711929321289,
    "flex": 1038.606460571289,
}
AGGREGATION_SIZE_MB: Dict[str, float] = {
    mode: DOWNLOAD_SIZE_MB[mode] + UPLOAD_SIZE_MB[mode] for mode in MODE_ORDER
}

# Figure-2 truncation controls
FIG2_TRUNCATE_FULL: bool = True
FIG2_TRUNCATE_MARGIN: float = 1.10


def comm_time_per_round_sec(mode: str) -> float:
    return DOWNLOAD_SIZE_MB[mode] / DOWNLINK_MBPS + UPLOAD_SIZE_MB[mode] / UPLINK_MBPS

def adjusted_cumulative_time_min(df: pd.DataFrame, mode: str) -> pd.Series:
    # round_metrics.csv logs pure compute (round_time / cumulative_time).
    # We add synthetic transmission cost per completed round.
    return (df["cumulative_time"].astype(float) + df["round"].astype(float) * comm_time_per_round_sec(mode)) / 60.0


print(f"Downlink: {DOWNLINK_MBPS:.2f} MB/s, uplink: {UPLINK_MBPS:.2f} MB/s")
for _mode in MODE_ORDER:
    print(
        f"  {_mode:4s}: down={DOWNLOAD_SIZE_MB[_mode]:8.2f} MB, "
        f"up={UPLOAD_SIZE_MB[_mode]:8.2f} MB, "
        f"total={AGGREGATION_SIZE_MB[_mode]:8.2f} MB, "
        f"comm/round={comm_time_per_round_sec(_mode):6.2f} s"
    )


In [ ]:
def load_round_metrics(dataset: str, mode: str) -> pd.DataFrame:
    csv_path = RESULTS_BASE / f"{dataset}_{mode}" / "round_metrics.csv"
    if not csv_path.exists():
        return pd.DataFrame()
    df = pd.read_csv(csv_path)
    required_cols = ["round", "eval_accuracy", "round_time", "cumulative_time"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column {col} in {csv_path}")
    df = df.sort_values("round").reset_index(drop=True)
    df["dataset"] = dataset
    df["mode"] = mode
    return df


all_results: Dict[str, Dict[str, pd.DataFrame]] = {}
for dataset in DATASET_ORDER:
    per_dataset: Dict[str, pd.DataFrame] = {}
    for mode in MODE_ORDER:
        df = load_round_metrics(dataset, mode)
        if not df.empty:
            per_dataset[mode] = df
    if per_dataset:
        all_results[dataset] = per_dataset

if not all_results:
    raise RuntimeError(f"No round_metrics.csv found under {RESULTS_BASE}")

available_datasets = [d for d in DATASET_ORDER if d in all_results]
print("Loaded datasets:", available_datasets)
for d in available_datasets:
    available_modes = list(all_results[d].keys())
    print(f"  {d}: {available_modes}")

In [ ]:
summary_rows = []

for dataset in available_datasets:
    wall_full_min = None

    for mode in MODE_ORDER:
        if mode not in all_results[dataset]:
            continue
        df = all_results[dataset][mode]
        final_acc = float(df["eval_accuracy"].iloc[-1])
        best_acc = float(df["eval_accuracy"].max())

        compute_time_min = float(df["cumulative_time"].iloc[-1] / 60.0)
        rounds_completed = int(float(df["round"].iloc[-1]))
        comm_time_min = rounds_completed * comm_time_per_round_sec(mode) / 60.0
        total_wall_time_min = compute_time_min + comm_time_min

        avg_round_compute_sec = float(df.loc[df["round"] >= 1, "round_time"].mean())
        comm_per_round_sec = float(comm_time_per_round_sec(mode))

        summary_rows.append(
            {
                "dataset": dataset,
                "dataset_label": DATASET_LABELS.get(dataset, dataset),
                "method": mode,
                "method_label": METHOD_LABELS[mode],
                "final_accuracy": final_acc,
                "best_accuracy": best_acc,
                "final_accuracy_pct": final_acc * 100.0,
                "best_accuracy_pct": best_acc * 100.0,
                "compute_time_min": compute_time_min,
                "comm_time_min": comm_time_min,
                "total_wall_time_min": total_wall_time_min,
                "avg_round_compute_sec": avg_round_compute_sec,
                "comm_per_round_sec": comm_per_round_sec,
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(
    ["dataset", "method"],
    key=lambda s: s.map({"20news": 0, "agnews": 1, "emotion": 2, "full": 0, "mix": 1, "drop": 2}).fillna(99),
)

# Speedup is based on total wall-clock time (compute + communication).
speedup_vals = []
for _, row in summary_df.iterrows():
    dataset = row["dataset"]
    mode = row["method"]
    full_rows = summary_df[(summary_df["dataset"] == dataset) & (summary_df["method"] == "full")]
    if mode == "full" or full_rows.empty:
        speedup_vals.append(None)
    else:
        speedup_vals.append(float(full_rows["total_wall_time_min"].iloc[0]) / float(row["total_wall_time_min"]))
summary_df["speedup_vs_full"] = speedup_vals

display_cols = [
    "dataset_label",
    "method_label",
    "final_accuracy_pct",
    "best_accuracy_pct",
    "compute_time_min",
    "comm_time_min",
    "total_wall_time_min",
    "speedup_vs_full",
]

print("Main End-to-End Results (Wall-clock includes manual transmission time)")
print(summary_df[display_cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

summary_csv = TABLE_DIR / "end_to_end_main_results.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"\nSaved main results CSV: {summary_csv}")

## Figure 1: Accuracy vs Communication Rounds


In [ ]:
def plot_accuracy_vs_rounds(results: Dict[str, Dict[str, pd.DataFrame]], save_path: Path) -> None:
    n = len(available_datasets)
    fig, axes = plt.subplots(1, n, figsize=(4.1 * n, 3.5), constrained_layout=True)
    if n == 1:
        axes = [axes]

    for idx, dataset in enumerate(available_datasets):
        ax = axes[idx]
        for mode in MODE_ORDER:
            if mode not in results[dataset]:
                continue
            df = results[dataset][mode]
            plot_df = df[df["round"] >= 1]
            ax.plot(
                plot_df["round"],
                plot_df["eval_accuracy"] * 100.0,
                label=METHOD_LABELS[mode],
                color=METHOD_COLORS[mode],
                marker=METHOD_MARKERS[mode],
                markersize=4,
                markevery=max(1, len(plot_df) // 12),
                linewidth=2.0,
            )

        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xlabel("Communication Round")
        if idx == 0:
            ax.set_ylabel("Test Accuracy (%)")
        ax.set_ylim(bottom=0)
        ax.grid(True, linestyle=":", alpha=0.4)
        ax.set_axisbelow(True)

        if idx == n - 1:
            ax.legend(frameon=False, loc="lower right")

    fig.savefig(save_path, bbox_inches="tight", format="pdf")
    plt.show()


fig1_path = FIG_DIR / "figure1_accuracy_vs_rounds.pdf"
plot_accuracy_vs_rounds(all_results, fig1_path)
print(f"Saved Figure 1: {fig1_path}")

## Figure 2: Accuracy vs Wall-clock Time

This figure uses adjusted wall-clock time:

- `adjusted_time = compute_time + round * transmission_time_per_round`
- `transmission_time_per_round = download_size_mb / downlink_mbps + upload_size_mb / uplink_mbps`

For readability, the x-axis can truncate the long `FedAvg-Full` tail.



In [ ]:
def plot_accuracy_vs_time(results: Dict[str, Dict[str, pd.DataFrame]], save_path: Path) -> None:
    n = len(available_datasets)
    fig, axes = plt.subplots(1, n, figsize=(4.1 * n, 3.5), constrained_layout=True)
    if n == 1:
        axes = [axes]

    for idx, dataset in enumerate(available_datasets):
        ax: plt.Axes = axes[idx]

        # Compute adjusted end time per method first (for truncation and speedup annotation)
        end_times: Dict[str, float] = {}
        for mode in MODE_ORDER:
            if mode not in results[dataset]:
                continue
            df = results[dataset][mode]
            plot_df = df[df["round"] >= 1]
            if not plot_df.empty:
                end_times[mode] = float(adjusted_cumulative_time_min(plot_df, mode).iloc[-1])

        x_limit = None
        if FIG2_TRUNCATE_FULL and "full" in end_times:
            non_full_ends = [v for k, v in end_times.items() if k != "full"]
            if non_full_ends:
                x_limit = max(non_full_ends) * FIG2_TRUNCATE_MARGIN

        for mode in MODE_ORDER:
            if mode not in results[dataset]:
                continue
            df = results[dataset][mode]
            plot_df = df[df["round"] >= 1]
            x_time = adjusted_cumulative_time_min(plot_df, mode)
            y_acc = plot_df["eval_accuracy"] * 100.0

            if x_limit is not None and mode == "full":
                visible_mask = x_time <= x_limit
                if visible_mask.any():
                    x_plot = x_time[visible_mask]
                    y_plot = y_acc[visible_mask]
                    ax.plot(
                        x_plot,
                        y_plot,
                        label=METHOD_LABELS[mode],
                        color=METHOD_COLORS[mode],
                        marker=METHOD_MARKERS[mode],
                        markersize=4,
                        markevery=max(1, len(x_plot) // 12),
                        linewidth=2.0,
                    )
                    if (~visible_mask).any():
                        ax.text(
                            0.98,
                            0.85,
                            f"Full truncated\nend={x_time.iloc[-1]:.1f} min",
                            transform=ax.transAxes,
                            ha="right",
                            va="top",
                            fontsize=8,
                            bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
                        )
                else:
                    # If full is entirely outside the visible x-range, keep legend entry with an empty plot.
                    ax.plot([], [], label=METHOD_LABELS[mode], color=METHOD_COLORS[mode])
            else:
                ax.plot(
                    x_time,
                    y_acc,
                    label=METHOD_LABELS[mode],
                    color=METHOD_COLORS[mode],
                    marker=METHOD_MARKERS[mode],
                    markersize=4,
                    markevery=max(1, len(x_time) // 12),
                    linewidth=2.0,
                )

        if x_limit is not None:
            ax.set_xlim(left=0.0, right=x_limit)

        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xlabel("Wall-clock Time (min)")
        if idx == 0:
            ax.set_ylabel("Test Accuracy (%)")
        ax.set_ylim(bottom=0)
        ax.grid(True, linestyle=":", alpha=0.4)
        ax.set_axisbelow(True)

        # if "full" in end_times and "mix" in end_times and end_times["mix"] > 0:
        #     speedup = end_times["full"] / end_times["mix"]
        #     ax.text(
        #         0.98,
        #         0.05,
        #         f"Mix speedup: {speedup:.2f}x",
        #         transform=ax.transAxes,
        #         ha="right",
        #         va="bottom",
        #         fontsize=8,
        #         bbox=dict(boxstyle="round", facecolor="white", alpha=0.7),
        #     )

        if idx == n - 1:
            ax.legend(frameon=False, loc="lower right")

    fig.savefig(save_path, bbox_inches="tight", format="pdf")
    plt.show()


fig2_path = FIG_DIR / "figure2_accuracy_vs_time.pdf"
plot_accuracy_vs_time(all_results, fig2_path)
print(f"Saved Figure 2: {fig2_path}")

In [ ]:
print("Generated artifacts:")
print(f"- Figure 1 PDF: {FIG_DIR / 'figure1_accuracy_vs_rounds.pdf'}")
print(f"- Figure 2 PDF: {FIG_DIR / 'figure2_accuracy_vs_time.pdf'}")
print(f"- Main results CSV: {TABLE_DIR / 'end_to_end_main_results.csv'}")